# 🎨 Pyxel Canvas
### 100 Stunning 2D & 3D Patterns in a Single Jupyter Notebook

---

Welcome to **Pyxel Canvas** — a fully interactive visual engine and gallery system.
Select a category, pick a pattern, adjust the controls, and click **RENDER**.

> **Tip:** Run all cells from top to bottom on first launch. Section 0 will verify all dependencies.

---
## Section 0 — Environment Setup
Verify all required libraries are available.

In [ ]:
# ── Section 0: Environment Setup ─────────────────────────────────
import importlib
from pathlib import Path

# Root path — all other paths are relative to this
NOTEBOOK_ROOT = Path.cwd()
print(f"📁 NOTEBOOK_ROOT = {NOTEBOOK_ROOT}")

# Required packages: (pip_name, import_name)
_REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("Pillow", "PIL"),
    ("ipywidgets", "ipywidgets"),
    ("scipy", "scipy"),
    ("tqdm", "tqdm"),
]

print("\n🔧 Checking dependencies...\n")
_missing = []
for pip_name, import_name in _REQUIRED:
    try:
        importlib.import_module(import_name)
        print(f"  ✅ {pip_name} — available")
    except ImportError:
        print(f"  ❌ {pip_name} — MISSING")
        _missing.append(pip_name)

if _missing:
    print(f"\n⚠️  Missing packages: {', '.join(_missing)}")
    print("   Install via MSYS2:  pacman -S mingw-w64-ucrt-x86_64-python-<name>")
    print("   Or via pip:         python -m pip install <name>")
else:
    # Create output directories
    for d in ["exports", "assets", "shaders"]:
        (NOTEBOOK_ROOT / d).mkdir(exist_ok=True)
    print("\n✨ Environment ready!")

---
## Section 1 — Core Engine
Load pattern registry, engines, and utilities into the notebook namespace.

In [ ]:
# ── Section 1: Core Engine ───────────────────────────────────────
import sys
from pathlib import Path

# Make sure our package is importable
_engine_root = str(Path.cwd().parent) if Path.cwd().name == 'visual_engine' else str(Path.cwd())
if _engine_root not in sys.path:
    sys.path.insert(0, _engine_root)

_ve_root = str(Path.cwd()) if Path.cwd().name == 'visual_engine' else str(Path.cwd() / 'visual_engine')
if _ve_root not in sys.path:
    sys.path.insert(0, _ve_root)

# Import the pattern registry
from patterns import PATTERNS, CATEGORIES
from engines import PALETTES
from engines.color_utils import ColorUtils
from utils.export import export_png

print(f"🎨 Loaded {len(PATTERNS)} patterns across {len(CATEGORIES)} categories")
for cat, pats in CATEGORIES.items():
    print(f"   • {cat}: {len(pats)} patterns")

---
## Section 2 — UI Controls
Interactive gallery interface — select a category and pattern, adjust controls, render.

In [ ]:
# ── Section 2: UI Controls ───────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

# ── Caches ────────────────────────────────────────────────────────
_render_cache   = {}   # renderer instances, keyed by pattern name
_controls_cache = {}   # widget lists, keyed by pattern name — same instances reused everywhere

def _get_renderer(name):
    if name not in _render_cache:
        _render_cache[name] = PATTERNS[name]
    return _render_cache[name]

def _get_controls(name):
    """Return the cached control widgets for a pattern (created once, reused forever)."""
    if name not in _controls_cache:
        _controls_cache[name] = _get_renderer(name).get_controls()
    return _controls_cache[name]

# ── Widgets ────────────────────────────────────────────────────────

category_dd = widgets.Dropdown(
    options=list(CATEGORIES.keys()),
    value=list(CATEGORIES.keys())[0],
    description='Category:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='320px'),
)

pattern_dd = widgets.Dropdown(
    options=CATEGORIES[category_dd.value],
    description='Pattern:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='320px'),
)

palette_dd = widgets.Dropdown(
    options=list(PALETTES.keys()),
    value='Inferno',
    description='Palette:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='220px'),
)

speed_slider = widgets.FloatSlider(
    value=1.0, min=0.1, max=5.0, step=0.1,
    description='Speed:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='220px'),
    readout_format='.1f',
)

resolution_dd = widgets.Dropdown(
    options=['Low', 'Medium', 'High'],
    value='Low',
    description='Resolution:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='220px'),
)

render_btn = widgets.Button(
    description='🎨 RENDER',
    button_style='success',
    layout=widgets.Layout(width='160px', height='40px'),
    style={'font_weight': 'bold'},
)

export_btn = widgets.Button(
    description='💾 EXPORT',
    button_style='info',
    layout=widgets.Layout(width='160px', height='40px'),
    style={'font_weight': 'bold'},
)

# VBox instead of Output — children replacement is atomic, no clear_output race conditions
pattern_controls_area = widgets.VBox(
    layout=widgets.Layout(
        border='1px solid #333', min_height='40px', padding='8px', margin='4px 0',
    )
)

render_output = widgets.Output(layout=widgets.Layout(
    border='1px solid #444', min_height='200px', padding='8px',
))

status_label = widgets.HTML(
    value='<i style="color:#888;">Select a pattern and click RENDER</i>',
    layout=widgets.Layout(margin='4px 0'),
)

# ── Callbacks ──────────────────────────────────────────────────────

def _on_category_change(change):
    """Switch category — update pattern_dd without double-firing _on_pattern_change."""
    pattern_dd.unobserve(_on_pattern_change, names='value')
    pattern_dd.options = CATEGORIES[change['new']]
    pattern_dd.value   = CATEGORIES[change['new']][0]
    pattern_dd.observe(_on_pattern_change, names='value')
    _on_pattern_change({'new': pattern_dd.value})   # fire exactly once

def _on_pattern_change(change):
    """Swap the controls panel — atomic VBox.children replace, no flicker or doubling."""
    controls = _get_controls(change['new'])
    pattern_controls_area.children = (
        controls if controls
        else [widgets.Label('No pattern-specific controls.')]
    )

_last_fig = [None]

def _on_render(btn):
    """Render selected pattern — plt.show() inside render() is the only display event."""
    with render_output:
        clear_output(wait=True)
        name = pattern_dd.value
        status_label.value = f'<b style="color:#4fc3f7;">⏳ Rendering: {name}...</b>'
        renderer = _get_renderer(name)

        # Read values from the DISPLAYED controls (same cached instances the user adjusted).
        extra_kwargs = {}
        for ctrl in _get_controls(name):
            if hasattr(ctrl, 'description') and hasattr(ctrl, 'value'):
                key = ctrl.description.strip(':').strip().lower().replace(' ', '_')
                extra_kwargs[key] = ctrl.value

        try:
            # render() calls plt.show() then plt.close(fig) — that is the one and only
            # display event.  No sink, no manual display() call needed here.
            renderer.render(
                resolution=resolution_dd.value,
                palette=palette_dd.value,
                speed=speed_slider.value,
                **extra_kwargs,
            )
            # renderer._fig is assigned by _create_figure() and the Python object
            # remains valid for savefig() even after plt.close() deregisters it.
            _last_fig[0] = renderer._fig
            status_label.value = f'<b style="color:#81c784;">✅ Rendered: {name}</b>'
        except Exception as e:
            status_label.value = f'<b style="color:#e57373;">❌ Error: {e}</b>'
            import traceback
            traceback.print_exc()

def _on_export(btn):
    """Export the last rendered figure to PNG."""
    if _last_fig[0] is not None:
        path = export_png(_last_fig[0], pattern_dd.value)
        status_label.value = f'<b style="color:#81c784;">💾 Exported → {path}</b>'
    else:
        status_label.value = '<b style="color:#ffb74d;">⚠️ Nothing to export — render a pattern first.</b>'

# Wire up callbacks
category_dd.observe(_on_category_change, names='value')
pattern_dd.observe(_on_pattern_change,   names='value')
render_btn.on_click(_on_render)
export_btn.on_click(_on_export)

# ── Layout ─────────────────────────────────────────────────────────

header = widgets.HTML(value='''
    <div style="background:linear-gradient(135deg,#1a1a2e,#16213e,#0f3460);
                padding:16px 24px; border-radius:12px; margin-bottom:12px;">
        <h2 style="color:#e0e0e0; margin:0;">🎨 Pyxel Canvas</h2>
        <p style="color:#888; margin:4px 0 0;">Select a category → pick a pattern → click RENDER</p>
    </div>
''')

selectors_row = widgets.HBox([category_dd, pattern_dd],
                              layout=widgets.Layout(gap='12px'))
controls_row  = widgets.HBox([palette_dd, speed_slider, resolution_dd],
                              layout=widgets.Layout(gap='12px'))
buttons_row   = widgets.HBox([render_btn, export_btn],
                              layout=widgets.Layout(gap='12px'))

ui = widgets.VBox([
    header,
    selectors_row,
    controls_row,
    pattern_controls_area,
    buttons_row,
    status_label,
    render_output,
], layout=widgets.Layout(padding='8px'))

display(ui)
_on_pattern_change({'new': pattern_dd.value})

---
## Section 3 — Render Section
The RENDER button above triggers rendering. No separate cell needed — the UI handles it.

---

## Section 4 — Pattern Implementations

All 100 patterns are implemented as classes in the `patterns/` module files.
Concept Notes for each completed pattern are listed below.

### How This Works — Mandelbrot Fractal Explorer

**Core Idea**  
The Mandelbrot set is the set of complex numbers $c$ for which the sequence produced by iterating $z_{n+1} = z_n^2 + c$ (starting from $z_0 = 0$) remains bounded — that is, never escapes to infinity. Points inside the set are colored black; points outside are colored by how quickly they escape, producing the iconic fractal boundary.

**Mathematics**  
The iteration is:
$$z_{n+1} = z_n^2 + c, \quad z_0 = 0, \quad c \in \mathbb{C}$$
Each pixel maps to a value of $c = x + iy$, where $x$ and $y$ are the real and imaginary axes. The escape condition is $|z_n| > 2$. The `max_iter` control sets the iteration ceiling: higher values reveal finer boundary detail at the cost of computation time. Smooth coloring uses the formula $n_{\text{smooth}} = n + 1 - \log_2(\log_2|z_n|)$ to eliminate banding.

**Logic in the Code**  
1. A 2D numpy array of complex numbers is constructed via `meshgrid` over the viewport defined by `center` and `zoom`.  
2. The iteration $z = z^2 + c$ is applied element-wise using numpy broadcasting — no Python loop over pixels.  
3. An escape mask tracks which pixels have crossed $|z| > 2$. Those pixels are frozen and excluded from subsequent steps.  
4. The smooth escape-count array is passed through a `ColorUtils` colormap to produce the final image.

**Interesting Properties**  
The Mandelbrot set is infinitely self-similar: zooming into any part of the boundary reveals structures that resemble — but are not identical to — the whole set. The cardioid shape of the main body is the locus of $c$ values where $z$ converges to a fixed point, while the circular bulb to its left corresponds to period-2 orbits.

### How This Works — Julia Set Animator

**Core Idea**  
A Julia set is the companion fractal to the Mandelbrot set. Instead of varying $c$ per pixel and starting from $z_0 = 0$, a Julia set fixes $c$ and varies the starting point $z_0$ across the complex plane. Each value of $c$ produces a different Julia set — connected (filled) when $c$ is inside the Mandelbrot set, and a Cantor dust when $c$ is outside.

**Mathematics**  
$$z_{n+1} = z_n^2 + c, \quad z_0 = x + iy \text{ (varies per pixel)}, \quad c \text{ is fixed}$$
The escape condition and smooth coloring are identical to Mandelbrot: $|z_n| > 2$ and $n_{\text{smooth}} = n + 1 - \log_2(\log_2|z_n|)$. The `c_real` and `c_imag` sliders control the fixed constant, allowing exploration of radically different fractal shapes.

**Logic in the Code**  
1. A complex grid of $z_0$ values is built from the viewport extent.  
2. The same vectorized escape-time loop runs, but with a constant $c$ added at each step.  
3. The resulting escape array is rendered through the selected colormap.

**Interesting Properties**  
Moving the $c$ parameter along the boundary of the Mandelbrot set produces Julia sets that transition from connected to disconnected — the so-called Douady rabbit ($c \approx -0.123 + 0.745i$) and the dendrite ($c = i$) are famous examples. The Mandelbrot set is, in fact, a map of all possible Julia set topologies.

### How This Works — Sierpinski Triangle

**Core Idea**  
The Sierpinski triangle is a self-similar fractal formed by recursively removing the central inverted triangle from an equilateral triangle. After infinite iterations the remaining area is zero, but the structure retains infinite detail. It appears naturally in Pascal's triangle (odd entries) and cellular automata (Rule 90).

**Mathematics**  
At each recursion level, every triangle is replaced by three sub-triangles at half scale, positioned at the three corners. The fractal dimension is:
$$D = \frac{\log 3}{\log 2} \approx 1.585$$
At depth $d$, the number of filled triangles is $3^d$ and each has side length $2^{-d}$ of the original. The `depth` control directly sets $d$.

**Logic in the Code**  
1. Start with the three vertices of an equilateral triangle.  
2. Recursively subdivide: compute midpoints of each edge, forming three corner triangles (discarding the center).  
3. At the base case ($d = 0$), store the triangle as a `[v0, v1, v2]` list.  
4. All triangles are drawn as filled `Polygon` patches with colors sampled from a gradient.

**Interesting Properties**  
The Sierpinski triangle can also be generated by the chaos game: repeatedly pick a random vertex and move halfway toward it from the current point. Despite the randomness, the attractor is deterministically the Sierpinski triangle.

### How This Works — Koch Snowflake

**Core Idea**  
The Koch snowflake is constructed by repeatedly replacing the middle third of every line segment with an equilateral triangular bump. Starting from an equilateral triangle, this process produces a closed curve of infinite length enclosing a finite area — a key early example of a fractal curve.

**Mathematics**  
Each iteration multiplies the number of segments by 4 and divides each segment's length by 3. After $n$ iterations:
$$L_n = L_0 \cdot \left(\frac{4}{3}\right)^n \to \infty, \qquad A_n = A_0 \cdot \frac{8}{5}\left(1 - \frac{3}{9^n}\right)$$
The fractal dimension of the Koch curve is $D = \log 4 / \log 3 \approx 1.262$. The `iterations` slider controls $n$.

**Logic in the Code**  
1. Define three vertices of an equilateral triangle as the snowflake base.  
2. For each edge, recursively subdivide: split into thirds, compute the peak point by rotating the middle third by $60°$ using a 2D rotation.  
3. Collect all points and draw each line segment with a color from the gradient, producing a rainbow Koch snowflake.

**Interesting Properties**  
The Koch snowflake is nowhere differentiable — it has no tangent at any point. It was one of the first curves shown to be continuous everywhere but differentiable nowhere, predating Mandelbrot's formalization of fractal geometry by decades.

### How This Works — Penrose Tiling

**Core Idea**  
A Penrose tiling is an aperiodic tiling — it fills the plane without gaps or overlaps, yet never repeats. Discovered by Roger Penrose in 1974, the P3 variant uses two rhombus shapes (thin and thick) derived from Robinson triangles. The pattern exhibits five-fold rotational symmetry, which is forbidden in periodic crystals.

**Mathematics**  
The construction uses the golden ratio $\varphi = (1 + \sqrt{5})/2 \approx 1.618$. Robinson triangle deflation splits each triangle into smaller ones using subdivision points at $1/\varphi$ along edges:
$$P = A + \frac{B - A}{\varphi}, \qquad Q = B + \frac{A - B}{\varphi}, \qquad R = B + \frac{C - B}{\varphi}$$
Thin (acute) triangles split into 2 sub-triangles; thick (obtuse) triangles split into 3. The `generations` control sets the number of deflation rounds.

**Logic in the Code**  
1. Start with a sun configuration: 10 isosceles triangles arranged in a decagon (alternating orientations).  
2. For each generation, apply the deflation rules — thin → 2 children, thick → 3 children.  
3. Draw all resulting triangles as filled polygons, using two palette colors to distinguish thin from thick rhombi.

**Interesting Properties**  
Penrose tilings are quasicrystals in 2D: they have long-range order (sharp diffraction peaks with five-fold symmetry) but no translational periodicity. The ratio of thick to thin rhombi converges to $\varphi$ as the tiling grows — the golden ratio appears at every scale.

### How This Works — Voronoi Diagram

**Core Idea**
A Voronoi diagram partitions a plane into regions based on proximity to a set of seed points. Each region — called a Voronoi cell — contains every point closer to its seed than to any other. The result is a natural tessellation found everywhere in nature: giraffe skin, soap bubble clusters, and cellular tissue.

**Mathematics**
For seeds $P = \{p_1, \ldots, p_n\}$, the Voronoi cell of $p_i$ is:
$$V(p_i) = \{ x \in \mathbb{R}^2 \mid \|x - p_i\| \leq \|x - p_j\| \;\forall\, j \neq i \}$$
The boundary between two adjacent cells is the perpendicular bisector of the segment joining their seeds. The *Points* slider controls $n$; more seeds produce smaller, more uniform cells.

**Logic in the Code**
1. $n$ random seeds are placed in $[0,1]^2$ using a seeded NumPy RNG for reproducibility.
2. A `scipy.spatial.cKDTree` spatial index is built on the seeds.
3. The canvas is rasterized as a flat grid of $\text{res}^2$ pixels; one `tree.query` call assigns the nearest seed to every pixel simultaneously — no Python loop over pixels.
4. The integer assignment array indexes into the palette gradient to produce the final RGB image rendered via `imshow`.

**Interesting Properties**
Voronoi diagrams are the geometric dual of Delaunay triangulations: connecting adjacent seeds whose cells share a border produces the Delaunay graph, which maximises the minimum triangle angle across the plane. Lloyd's algorithm — iteratively replacing each seed with its cell centroid — converges toward perfectly uniform hexagonal packing.

### How This Works — Fibonacci Spiral

**Core Idea**
The Fibonacci spiral places $n$ points by rotating each successive point by the golden angle — the strategy used by sunflower seeds, pine cones, and pineapple scales to pack the maximum number of seeds per unit area without gaps or clumping.

**Mathematics**
Each seed $i$ is placed at polar coordinates:
$$r_i = \sqrt{i/n}, \qquad \theta_i = i \cdot \phi_g$$
where the **golden angle** is:
$$\phi_g = \pi(3 - \sqrt{5}) \approx 137.508°$$
The irrational nature of $\phi_g$ — derived from the golden ratio $\varphi = \tfrac{1+\sqrt{5}}{2}$ — ensures no two seeds ever land on the same radial spoke, so every new seed fills the largest visible gap.

**Logic in the Code**
1. Indices $i = 0, \ldots, n{-}1$ are generated as a NumPy array.
2. Radii are $\sqrt{i/n}$ (uniform area distribution) and angles are $i \cdot \phi_g$.
3. Cartesian coordinates follow from standard polar conversion; the palette gradient maps $i$ linearly to colour so inner and outer seeds receive different hues.
4. `ax.scatter` renders all $n$ points in a single vectorised call.

**Interesting Properties**
Plotting consecutive Fibonacci-number multiples of seeds reveals the clockwise and counter-clockwise spiral arms visible in real sunflowers. Displacing the angle by even 0.1° away from $\phi_g$ causes the pattern to collapse into coarse radial spokes — demonstrating why the golden angle is the uniquely optimal packing angle.

### How This Works — Dragon Curve

**Core Idea**
The Dragon Curve is a fractal obtained by repeatedly folding a strip of paper in half in the same direction, then unfolding it so every crease stands at 90°. After $n$ folds the unfolded boundary traces a self-avoiding curve that tiles the plane when four copies are assembled around a common point.

**Mathematics**
The turn sequence $T_n$ — recording right (R) or left (L) turns at each step — obeys the recurrence:
$$T_{n+1} = T_n \;\|\; [\text{R}] \;\|\; \overline{T_n^{\;\text{rev}}}$$
where $\overline{\cdot}$ flips all turns (R$\leftrightarrow$L) and $\cdot^{\text{rev}}$ reverses the list. Starting from $T_1 = [\text{R}]$, iteration $n$ yields $2^n - 1$ turns and $2^n$ unit segments. The Hausdorff dimension of the Dragon Curve boundary is $\tfrac{\log 4}{\log 3} \approx 1.52$.

**Logic in the Code**
1. The turn list is built iteratively: each pass appends the current list, a right turn, then the reversed-and-negated list.
2. A direction index (0–3, cycling right/up/left/down) is updated for each turn and used to step the path one unit forward.
3. The path array is converted into a `(N{-}1, 2, 2)` segment array and passed to `LineCollection` — all $2^n{-}1$ segments drawn in a single renderer call, essential for $n \geq 12$.

**Interesting Properties**
Despite its jagged appearance the Dragon Curve never self-intersects, yet in the limit $n \to \infty$ it fills a bounded fractal region completely. Four copies rotated by multiples of 90° tile the plane perfectly — a property exploited by Jurassic Park's book cover art and numerous fractal installations.

### How This Works — Hilbert Curve

**Core Idea**
The Hilbert Curve is a continuous space-filling curve that visits every cell of an $n \times n$ grid exactly once while moving only to directly adjacent cells. Invented by David Hilbert in 1891, it preserves spatial locality so well that nearby points in 2D remain nearby along the 1D index — making it invaluable for database locality, image compression, and CPU cache optimisation.

**Mathematics**
At order $k$ the curve covers a $2^k \times 2^k$ grid visiting all $4^k$ cells. Each order is built from four rotated/reflected copies of the previous order. The index-to-coordinate mapping decodes $d \in [0,4^k)$ by reading successive 2-bit pairs $(r_x, r_y)$:
$$r_x = \bigl\lfloor d/2 \bigr\rfloor \bmod 2, \quad r_y = (r_x \oplus d) \bmod 2$$
with a conditional 90° rotation applied at each level before accumulating the grid offset.

**Logic in the Code**
1. `_hilbert_xy` loops over all $4^k$ indices; for each it iterates over $k$ bit-pair levels, applying the rotation and offset to accumulate $(x, y)$.
2. The resulting $(4^k, 2)$ coordinate array is stacked into $(4^k{-}1, 2, 2)$ segments for a `LineCollection` draw — no individual `ax.plot` calls.
3. The palette gradient maps position along the 1D index to colour, so the progression of the space-filling traversal is immediately visible.

**Interesting Properties**
The Hilbert Curve's locality preservation is quantified: the Euclidean distance between two grid cells never exceeds $O(\sqrt{\Delta d})$ where $\Delta d$ is their 1D index distance — far superior to the $O(n)$ worst case of a row-scan. Higher orders reveal the same rotated U-shape repeating at every scale, a canonical example of exact geometric self-similarity.

---

## Section 5 — Export Utilities

Export functions are available via the **💾 EXPORT** button above.
- `export_png(figure, name)` — saves as PNG
- `export_gif(frames, name, fps)` — saves as animated GIF
- `export_mp4(frames, name, fps)` — saves as MP4 (falls back to GIF if ffmpeg unavailable)

All outputs are saved to `visual_engine/exports/`.